# **Notebook 2a: Extracting features**

**Feature categories:**
1. Basic lexical (length, punctuation, pronouns)
2. Lexicon matches (hyperbolic, evaluative, intensifiers, success markers, comparison, imperatives)
3. Syntactic via spaCy (POS distribution, adjective stacking)
4. Named entities via spaCy (organizations, persons, awards)
5. Position in blurb
6. Context-relative (sentence vs rest of blurb)
7. Reference-based (sentence vs final_edit) — most novel feature direction
8. Token-level (token count via BERTje tokenizer)

In [9]:
import json

notebook_path = "your_notebook.ipynb"

with open(notebook_path, 'r') as f:
    nb = json.load(f)

# Zorg dat metadata.widgets.state bestaat
if 'metadata' not in nb:
    nb['metadata'] = {}
if 'widgets' not in nb['metadata']:
    nb['metadata']['widgets'] = {}
if 'state' not in nb['metadata']['widgets']:
    nb['metadata']['widgets']['state'] = {}

with open(notebook_path, 'w') as f:
    json.dump(nb, f, indent=1)

print(f"Fixed: {notebook_path}")

FileNotFoundError: [Errno 2] No such file or directory: 'your_notebook.ipynb'

## 1. Installs

In [ ]:
!pip install -q sentence-transformers
!python -m spacy download nl_core_news_lg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 568.1/568.1 MB 976.0 kB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('nl_core_news_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


## 2. Importing the packages

In [ ]:
import os
import re
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import spacy
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer

Mounted at /content/drive
Working dir: /content/drive/MyDrive/Individual thesis part/Model 2: Feature based


## 3. Preparation

In [ ]:
# Input paths
TRAIN_CSV = "Data/labeled_train.csv"
VAL_CSV = "Data/labeled_val.csv"
TEST_CSV = "Data/labeled_test.csv"
UNLABELED_TSV = "unlabeled_data.tsv"

# Output paths
OUTPUT_DIR = "./Data_with_features"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Column names
SENT_COL = "blurb_sent"
BLURB_COL = "blurb"
FINAL_EDIT_COL = "final_edit"
BLURB_ID_COL = "blurb_id"
POSITION_COL = "sent_position"
LABEL_COL = "unnecessary"    # only present in labeled data

# Model names
SPACY_MODEL = "nl_core_news_lg"
SBERT_MODEL = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
BERTJE_MODEL = "GroNLP/bert-base-dutch-cased"

## 4. Loading needed models

In [ ]:
nlp = spacy.load(SPACY_MODEL)
sbert = SentenceTransformer(SBERT_MODEL)
bertje_tokenizer = AutoTokenizer.from_pretrained(BERTJE_MODEL)
print("Models loaded.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.12k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/254 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/242k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Models loaded.


## 5. Dutch lexicons

In [ ]:
# Note: these lists were started by me and extended using Claude (AI), then reviewed and approved by me

LEXICONS = {
    # PROMOTIONAL ADJECTIVES (grouped by linguistic relatedness)
    # --------------------------------------------------------------------------
    "promotional_hyperbolic": [
        # Exaggerated scale and intensity claims
        "ongeëvenaard", "ongeevenaard", "revolutionair", "baanbrekend",
        "ongekend", "weergaloos", "legendarisch", "iconisch", "historisch",
        "monumentaal", "wereldschokkend", "ontzagwekkend", "tijdloos",
        "essentieel", "onmisbaar", "klassieker", "moderne klassieker",
        "instant klassieker", "verbluffend", "adembenemend", "duizelingwekkend", "overweldigend",
        "verpletterend", "verbazingwekkend", "verstommend",
        "explosief", "explosieve", "huiveringwekkend", "ademloos",
        "hypnotiserend", "ongelofelijk", "ongelooflijk",],

    "promotional_evaluative": [
        # Generic evaluative adjectives covering quality, emotion, style, and tone
        "prachtig", "schitterend", "geweldig", "perfect", "magnifiek",
        "voortreffelijk", "uitmuntend", "verrukkelijk", "wonderschoon",
        "subliem", "magistraal", "geniaal", "briljant", "fenomenaal",
        "indrukwekkend", "meesterlijk", "meeslepend", "onvergetelijk",
        "aangrijpend", "hartverscheurend", "betoverend", "fascinerend",
        "ontroerend", "schrijnend", "ontwapenend", "indringend", "doorleefd",
        "intens", "diepgaand", "hartveroverend", "onthutsend", "ontluisterend",
        "verheffend", "verfijnd", "subtiel", "geestig", "humoristisch",
        "scherpzinnig", "vlijmscherp", "scherp", "verfrissend", "gedurfd",
        "moedig", "origineel", "vernieuwend", "trefzeker", "raak", "rauw",
        "rauwe", "ongekunsteld", "puur", "authentiek", "compromisloos", "warm",
        "warmhartig", "liefdevol", "tedere", "teder", "vurig", "gepassioneerd",
        "hartstochtelijk", "speels", "gevoelig", "wijs", "wrang", "krachtig",
        "magisch", "mythisch", "wonderlijk", "ongewoon",],


    # INTENSIFIERS
    # --------------------------------------------------------------------------
    "intensifier": [
        "zeer", "heel", "erg", "uiterst", "absoluut", "buitengewoon",
        "totaal", "volkomen", "volledig", "enorm", "extreem", "bijzonder",
        "ontzettend", "verschrikkelijk", "ongelofelijk", "uitermate",
        "razend", "verbazend", "uitzonderlijk", "buitensporig",
        "uitgesproken", "compleet", "puur", "louter", "werkelijk",
        "ronduit", "regelrecht", "bovenmate", "extra",
        "buitenproportioneel",],


    # SUCCESS MARKER
    # --------------------------------------------------------------------------
    "success_marker": [
        # Bestseller variants
        "bestseller", "new york times bestseller", "nyt bestseller",
        "sunday times bestseller", "bestsellerlijst", "internationale bestseller",
        "wereldbestseller", "#1 bestseller", "nummer 1 bestseller",
        "nr. 1", "nr 1", "top 10", "top tien", "top 100",
        "bestverkochte", "bestverkocht", "topverkoper", "verkooptopper", "must-read",
        "must read", "verplichte kost", "essentiële lectuur", "een must",
        # Sales / reach
        "miljoen exemplaren", "miljoenen exemplaren",
        "honderdduizenden exemplaren", "exemplaren verkocht",
        "vertaald in", "in meer dan", "in tientallen landen",
        "wereldwijd", "internationaal", "wereldsucces",
        # Recognition
        "genomineerd", "bekroond", "winnaar", "prijswinnend",
        "succesvol", "succesvolle", "befaamd", "geroemd", "geprezen",
        "doorbraak", "literaire sensatie", "lovende kritieken",
        "lovende recensies", "unaniem geprezen", "kritisch geprezen",],


    # AWARDS
    # --------------------------------------------------------------------------
    "award_keyword": [
        # General
        "prijs", "award", "medaille", "onderscheiding", "bekroning",
        "literatuurprijs", "boekenprijs", "nominatie", "winnaar van",
        "genomineerd voor", "eerbetoon", "erevermelding",
        # Specific Dutch / Belgian awards
        "ako literatuurprijs", "libris literatuurprijs",
        "boekenweekgeschenk", "gouden uil", "constantijn huygens-prijs",
        "p.c. hooft-prijs", "anv debutantenprijs", "anton wachterprijs",
        "fintro literatuurprijs", "vlaamse cultuurprijs",],


    # COMPARISONS
    # --------------------------------------------------------------------------
    "comparison_marker": [
        # Comparisons to other works or authors
        # (formerly: comparison_to_works)
        "net als", "in de traditie van", "vergelijkbaar met",
        "doet denken aan", "in de geest van", "vergelijk",
        "vergelijkbaar", "in de lijn van", "in navolging van",
        "lijkt op", "herinnert aan", "qua stijl", "soortgelijk",
        "à la", "in de stijl van", "geïnspireerd door",
        "geinspireerd door", "evenals", "ontmoet", "kruising tussen",
        "een mix van", "een combinatie van",
        # Direct address to the potential reader
        # (formerly: comparison_to_reader)
        "voor lezers van", "voor de liefhebbers van", "voor fans van",
        "voor wie houdt van", "iets voor", "ook geschikt voor",
        "zal aanspreken", "zal bevallen", "perfect voor", "ideaal voor",
        "valt in de smaak bij", "geliefd bij", "geliefd door", "geschreven voor",],


    # IMPERATIVES
    # --------------------------------------------------------------------------
    "imperative_action": [
        # Discovery / experience
        "ontdek", "verken", "verdiep", "leer", "stel je voor",
        "verbeeld je", "stap in", "treed binnen", "betreed",
        "beleef", "duik", "stort", "dompel", "geniet", "ervaar",
        "proef", "voel", "raak", "leef mee", "ga mee",
        "laat je meeslepen", "laat je verrassen", "laat je raken",
        "laat je betoveren", "laat je verleiden", "laat je",
        # Urgency / purchase
        "mis niet", "verlies", "vergeet niet", "wacht niet",
        "aarzel niet", "twijfel niet", "pak", "grijp", "koop",
        "bestel", "kies",],


    # SECOND PERSON
    # --------------------------------------------------------------------------
    "second_person": [
        # Pronouns
        "je", "jij", "jou", "jouw", "u", "uw", "jezelf", "jullie",
        # Common phrases
        "u zult", "je zult", "je kunt", "u kunt", "je moet", "u moet",
        "je gaat", "u gaat", "je wordt", "u wordt", "je voelt", "u voelt",
        "voor jou", "voor u",],


    # MEDIA OUTLETS
    # --------------------------------------------------------------------------
    "media_outlet": [
        # Dutch newspapers
        "nrc", "nrc handelsblad", "volkskrant", "de volkskrant",
        "trouw", "telegraaf", "de telegraaf", "parool", "het parool",
        "financieel dagblad", "fd", "ad", "algemeen dagblad", "metro",
        "nederlands dagblad", "reformatorisch dagblad",
        # Belgian newspapers
        "de morgen", "de standaard", "het laatste nieuws", "hln",
        "het nieuwsblad", "gazet van antwerpen", "de tijd",
        # Magazines
        "vrij nederland", "vn", "de groene", "de groene amsterdammer",
        "hp/de tijd", "humo", "knack", "elsevier", "ew magazine",
        "linda", "libelle", "margriet",
        # Literary outlets
        "literair nederland", "tzum", "athenaeum", "boekblad",
        "de reactor", "ons erfdeel", "dietsche warande",
        # Broadcasting
        "vpro", "nos", "npo", "radio 1", "klara", "canvas",
        "vrt", "vrt nws", "de wereld draait door",
        # International
        "new york times", "nyt", "the times", "the independent",],


    # CITATION MARKERS
    # --------------------------------------------------------------------------
    "citation_marker": [
        "aldus", "volgens", "schrijft", "schreef", "noemt het",
        "spreekt van", "in de woorden van", "zoals zegt",
        "stelt", "concludeert", "betoogt", "beweert", "verklaart",
        "verzekert", "constateert", "merkt op", "merkte op",
        "stelt vast", "wijst op", "zegt", "zei", "vindt", "vond",
        "meent", "meende", "oordeelt", "oordeelde",
        "in een recensie", "in zijn recensie", "in haar recensie",
        "lovende recensie", "kritische recensie",],


    # READER EXPERIENCE
    # --------------------------------------------------------------------------
    "reader_experience": [
        # Unputdownable claims
        "niet meer wegleggen", "in één adem", "in één ruk",
        "leest als een trein", "kan je niet wegleggen",
        "tot de laatste bladzijde", "boeit van begin tot eind",
        "houdt je in z'n greep", "houdt je in de ban",
        "pageturner", "page-turner", "page turner",
        "in één keer uitlezen", "uitgelezen", "verslindend",
        "verslindt", "verslindt je", "ademloze spanning",
        "tot het einde", "tot de laatste zin", "tot de laatste pagina",
        # Emotional aftermath
        "blijft je bij", "vergeet je nooit", "laat je niet los",
        "raakt je", "raakt diep", "blijft hangen", "blijft resoneren",
        "een ervaring", "een belevenis", "kippenvel",
        "tot tranen toe", "huiveringen",
        # Transportive
        "neemt je mee", "voert je mee", "trekt je in",
        "zuigt je in", "sleurt je", "een rollercoaster",
        "achtbaan", "emotionele rollercoaster",],


    # AUTHOR INFO
    # --------------------------------------------------------------------------
    "author_info": [
        # Functions and professions
        "auteur", "schrijver", "schrijfster", "romancier",
        "romanschrijver", "dichter", "dichteres", "essayist",
        "filosoof", "journalist", "journaliste", "vertaler",
        "vertaalster", "redacteur", "redactrice", "uitgever",
        "illustrator", "illustratrice", "tekenaar", "tekenares",
        "fotograaf", "fotografe", "samensteller", "samenstelster",
        "bezorgd door", "samengesteld door", "bewerkt door",
        "geredigeerd door", "ingeleid door", "voorwoord",
        "nawoord", "vertaald door",
        # Personal biographical information
        "geboren in", "geboren te", "geboren op", "groeide op",
        "groeide op in", "woont in", "woont en werkt",
        "leeft en werkt", "is gevestigd", "afkomstig uit",
        "studeerde", "studeerde aan", "promoveerde", "afgestudeerd",
        "behaalde", "doceerde", "hoogleraar", "professor",
        "docent", "verbonden aan", "is getrouwd", "vader van",
        "moeder van",
        # Previous works and career
        "debuteerde", "debuut", "eerste roman", "eerdere romans",
        "eerder werk", "vorige boek", "vorige roman", "vorig werk",
        "publiceerde eerder", "schreef eerder", "is auteur van",
        "is bekend van", "is bekend om", "publiceerde",
        "verscheen eerder", "eerder verschenen", "vorige werken",
        "bouwde een oeuvre",],

    # ============================================================
    # GENRE MARKERS
    # --------------------------------------------------------------------------
    "genre_marker": [
        "thriller", "psychologische thriller", "literaire thriller",
        "spannende roman", "spannend boek", "historische roman",
        "literaire roman", "klassieker", "modern klassiek",
        "feel-good roman", "feelgood", "feel good", "romcom",
        "chicklit", "young adult", "ya", "fantasy", "sci-fi",
        "science fiction", "horror", "dystopie", "memoir",
        "biografie", "autobiografie", "reisverhaal", "essaybundel",
        "verhalenbundel", "novelle", "kort verhaal",],}

## 6. Feature extraction functions

In [ ]:
# ------------------------------------------------------------------------------
# 6a. Basic lexical features
# ------------------------------------------------------------------------------
def basic_lexical_features(sent):
    """Length, punctuation, casing"""
    s = str(sent) if pd.notna(sent) else ""
    words = s.split()

    # Quote characters: only DOUBLE quotes are counted (straight + typographic).
    # Single quote variants (U+0027, U+2018, U+2019) are excluded because in Dutch
    # they predominantly mark apostrophes within words ('s ochtends, d'r) or single
    # word emphasis, not the reviewer-testimonial patterns I want to capture.
    # Used \u escapes to avoid copy-pasting flattening typographic to straight quotes.
    quote_chars = [
        '\u0022',   # "  straight double
        '\u201C',   # "  left double typographic
        '\u201D',   # "  right double typographic
        '\u201E',   # „  low double (occasional in Dutch typography)
]
    opening_quotes = tuple(quote_chars)

    return {
        "char_count":        len(s), # Amount of characters
        "word_count":        len(words), # Amount of words
        "avg_word_len":      np.mean([len(w) for w in words]) if words else 0, # Average word length per sentence per blurb
        "n_exclamation":     s.count("!"), # Amount of exclamation marks
        "n_question":        s.count("?"), # Amount of question marks
        "n_quote_chars":     sum(s.count(q) for q in quote_chars), # Amount of quote signs
        "n_uppercase_words": sum(1 for w in words if w.isupper() and len(w) > 1), # Amount of uppercase words
        "starts_with_quote": int(s.strip().startswith(opening_quotes)), # Binary: starts with quote mark
        "ends_with_dash":    int(s.strip().endswith("—") or s.strip().endswith("-")), # Binary: ends with em dash
        "has_em_dash":       int("—" in s),
    }

# ------------------------------------------------------------------------------
# 6b. Lexicon match features
# ------------------------------------------------------------------------------
def lexicon_features(sent):
    """Count per lexicon category."""
    s = str(sent).lower() if pd.notna(sent) else ""
    features = {}
    for category, terms in LEXICONS.items():
        count = sum(1 for term in terms if term in s)
        features[f"lex_{category}_count"] = count
    return features

# ------------------------------------------------------------------------------
# 6c. Syntactic features via spaCy
# ------------------------------------------------------------------------------
def syntactic_features(doc):
    """POS distribution, adjective stacking."""
    n_tokens = max(len(doc), 1)
    pos_counts = {"ADJ": 0, "ADV": 0, "VERB": 0, "AUX": 0, "NOUN": 0, "PROPN": 0, "PRON": 0}
    for token in doc:
        if token.pos_ in pos_counts:
            pos_counts[token.pos_] += 1

    # Adjective stacking: consecutive ADJ tokens, treating punctuation as transparent.
    # Without this, "meeslepende, ontroerende, onvergetelijke" would not be detected
    # as a stack because the commas would break it.
    adj_run = 0
    max_adj_run = 0
    n_adj_stacks = 0
    for token in doc:
        if token.pos_ == "ADJ":
            adj_run += 1
            max_adj_run = max(max_adj_run, adj_run)
        elif token.pos_ == "PUNCT":
            continue  # punctuation doesn't break the stack
        else:
            if adj_run >= 2:
                n_adj_stacks += 1
            adj_run = 0
    if adj_run >= 2:
        n_adj_stacks += 1

    # sentences starting with a verb
    starts_with_verb = int(len(doc) > 0 and doc[0].pos_ == "VERB")

    return {
        "n_tokens_spacy":   n_tokens,
        "n_adj":            pos_counts["ADJ"],
        "n_adv":            pos_counts["ADV"],
        "n_verb":           pos_counts["VERB"],
        "n_aux":            pos_counts["AUX"],
        "n_noun":           pos_counts["NOUN"],
        "n_propn":          pos_counts["PROPN"],
        "n_pron":           pos_counts["PRON"],
        "adj_density":      pos_counts["ADJ"] / n_tokens,
        "verb_density":     pos_counts["VERB"] / n_tokens,
        "aux_density":      pos_counts["AUX"] / n_tokens,
        "noun_density":     pos_counts["NOUN"] / n_tokens,
        "max_adj_run":      max_adj_run,
        "n_adj_stacks":     n_adj_stacks,
        "starts_with_verb": starts_with_verb,}

# ------------------------------------------------------------------------------
# 6d. Named entity features via spaCy
# ------------------------------------------------------------------------------
def ner_features(doc):
    """Count entities by type."""
    ent_counts = {"PERSON": 0, "ORG": 0, "LOC": 0, "GPE": 0, "NORP": 0, "EVENT": 0, "WORK_OF_ART": 0, "MISC": 0}
    for ent in doc.ents:
        if ent.label_ in ent_counts:
            ent_counts[ent.label_] += 1
    return {
        "n_person":      ent_counts["PERSON"],       # Persons names etc.
        "n_org":         ent_counts["ORG"],          # Organisation
        "n_loc":         ent_counts["LOC"],          # Non-geopolitical entities (e.g. beach)
        "n_gpe":         ent_counts["GPE"],          # geopolitical entity (countries, cities)
        "n_norp":        ent_counts["NORP"],         # nationalities, religious/political groups
        "n_event":       ent_counts["EVENT"],        # awards, festivals, events
        "n_workofart":   ent_counts["WORK_OF_ART"],  # book/film titles
        "n_misc":        ent_counts["MISC"],
        "total_ents":    sum(ent_counts.values()),
    }

# ------------------------------------------------------------------------------
# 6e. BERTje token count
# ------------------------------------------------------------------------------
def bertje_token_count(sent):
    s = str(sent) if pd.notna(sent) else ""
    if not s.strip():
        return {"n_bertje_tokens": 0}
    n = len(bertje_tokenizer.encode(s, add_special_tokens=True, truncation=False))
    return {"n_bertje_tokens": n}

## 7. Context-relative features (sentence vs blurb, sentence vs final_edit)
**Note:** sentence vs final_edit can and wil only be used for the weak label generator rather then the feature based baseline

In [ ]:
def encode_texts(texts, batch_size=64):
    """Encodes a list of texts with sentence transformer."""
    texts = [str(t) if pd.notna(t) else "" for t in texts]
    return sbert.encode(texts, batch_size=batch_size, show_progress_bar=True, convert_to_numpy=True)

def compute_context_features(df, has_final_edit=True):
    """
    Computes features that compare each sentence to:
    - The rest of its blurb
    - The final_edit
    Returns dict of feature columns.
    """
    # Encoding sentences
    sent_emb = encode_texts(df[SENT_COL].tolist())

    # For "rest of blurb", compute blurb embeddings excluding current sentence
    print("Computing similarity to rest of blurb...")
    rest_of_blurb_texts = []
    for blurb_id, group in df.groupby(BLURB_ID_COL, sort=False):
        sents_in_blurb = group[SENT_COL].tolist()
        for idx, sent in enumerate(sents_in_blurb):
            others = sents_in_blurb[:idx] + sents_in_blurb[idx+1:]
            rest_of_blurb_texts.append(" ".join(str(s) for s in others if pd.notna(s)))
    rest_emb = encode_texts(rest_of_blurb_texts)

    # Calculating cosine similarity per row
    sim_to_rest = np.array([
        cosine_similarity([sent_emb[i]], [rest_emb[i]])[0, 0] if np.linalg.norm(rest_emb[i]) > 0 else 0.0
        for i in range(len(sent_emb))])

    out = {"sim_to_rest_of_blurb": sim_to_rest}

    if has_final_edit and FINAL_EDIT_COL in df.columns:
        print("Computing similarity to final_edit...")
        # final_edit is per blurb, so encoding unique blurbs and map back
        unique_blurbs = df.drop_duplicates(BLURB_ID_COL)[[BLURB_ID_COL, FINAL_EDIT_COL]]
        final_edit_emb = encode_texts(unique_blurbs[FINAL_EDIT_COL].tolist())
        blurb_to_emb = {bid: emb for bid, emb in zip(unique_blurbs[BLURB_ID_COL], final_edit_emb)}

        sim_to_final = []
        for i, bid in enumerate(df[BLURB_ID_COL].values):
            fe_emb = blurb_to_emb.get(bid)
            if fe_emb is not None and np.linalg.norm(fe_emb) > 0:
                sim_to_final.append(cosine_similarity([sent_emb[i]], [fe_emb])[0, 0])
            else:
                sim_to_final.append(0.0)
        out["sim_to_final_edit"] = np.array(sim_to_final)

        # Word overlap with final_edit
        print("Computing word overlap with final_edit...")

        # Pre-computing word sets per unique blurb only ONCE
        fe_words_per_blurb = {
            bid: set(re.findall(r"\w+", str(fe).lower()))
            for bid, fe in zip(unique_blurbs[BLURB_ID_COL], unique_blurbs[FINAL_EDIT_COL])}

        overlap_ratio = []
        for sent_text, bid in zip(df[SENT_COL].values, df[BLURB_ID_COL].values):
            sent_words = set(re.findall(r"\w+", str(sent_text).lower()))
            fe_words = fe_words_per_blurb.get(bid, set())
            if len(sent_words) > 0:
                overlap_ratio.append(len(sent_words & fe_words) / len(sent_words))
            else:
                overlap_ratio.append(0.0)
        out["word_overlap_final_edit"] = np.array(overlap_ratio)
    return out

## 8. Blurb-relative statistics (relative density features)

In [ ]:
def add_blurb_relative_features(df_features, df_meta):
    """
    Add features comparing the sentence's values to the blurb average.
    Negative = sentence below blurb average; positive = above.
    """
    df_combined = df_features.copy()
    df_combined[BLURB_ID_COL] = df_meta[BLURB_ID_COL].values

    cols_to_relativize = ["adj_density", "verb_density", "word_count", "n_adj"]

    for col in cols_to_relativize:
        if col not in df_combined.columns:
            continue
        blurb_means = df_combined.groupby(BLURB_ID_COL)[col].transform("mean")
        df_combined[f"{col}_rel_to_blurb"] = df_combined[col] - blurb_means

    df_combined = df_combined.drop(columns=[BLURB_ID_COL])
    return df_combined

## 9. Main extraction pipeline

In [ ]:
def extract_features_for_df(df, name="data"):
    """
    Run full feature extraction on a dataframe.
    Returns a new dataframe with all features + identifying columns + label (if present).
    """
    print(f"\n{'='*60}\nExtracting features for: {name}\n{'='*60}")
    print(f"Input: {len(df)} sentences")

    # Per-sentence features
    print("\nComputing per-sentence features (basic, lexicon, bertje tokens)...")
    rows = []
    for sent in tqdm(df[SENT_COL].tolist(), desc="sentence-level"):
        feats = {}
        feats.update(basic_lexical_features(sent))
        feats.update(lexicon_features(sent))
        feats.update(bertje_token_count(sent))
        rows.append(feats)
    basic_df = pd.DataFrame(rows)

    # SpaCy features (POS + NER) =====
    print("\nRunning spaCy pipeline...")
    spacy_rows = []
    texts = [str(s) if pd.notna(s) else "" for s in df[SENT_COL].tolist()]
    for doc in tqdm(nlp.pipe(texts, batch_size=64), total=len(texts), desc="spaCy"):
        d = {}
        d.update(syntactic_features(doc))
        d.update(ner_features(doc))
        spacy_rows.append(d)
    spacy_df = pd.DataFrame(spacy_rows)

    # Position features
    print("\nComputing position features...")
    if POSITION_COL in df.columns:
        position_df = pd.DataFrame({"sent_position": df[POSITION_COL].values})
        # Relative position within blurb
        blurb_max_pos = df.groupby(BLURB_ID_COL)[POSITION_COL].transform("max")
        position_df["rel_position"] = df[POSITION_COL].values / np.maximum(blurb_max_pos.values, 1)
        position_df["is_first"] = (df[POSITION_COL].values == df.groupby(BLURB_ID_COL)[POSITION_COL].transform("min").values).astype(int)
        position_df["is_last"]  = (df[POSITION_COL].values == blurb_max_pos.values).astype(int)
        position_df = position_df.reset_index(drop=True)
    else:
        position_df = pd.DataFrame()

    # Combining before context features
    features_df = pd.concat([basic_df.reset_index(drop=True),
                              spacy_df.reset_index(drop=True),
                              position_df], axis=1)

    # Context-relative features (semantic similarity)
    print("\nComputing context-relative features")
    context_feats = compute_context_features(df, has_final_edit=(FINAL_EDIT_COL in df.columns))
    for k, v in context_feats.items():
        features_df[k] = v

    # Blurb-relative features
    print("\nComputing blurb-relative features...")
    features_df = add_blurb_relative_features(features_df, df)

    # Add identifying columns and label (if present)
    features_df[BLURB_ID_COL] = df[BLURB_ID_COL].values
    features_df[SENT_COL] = df[SENT_COL].values
    if LABEL_COL in df.columns:
        features_df[LABEL_COL] = df[LABEL_COL].values

    print(f"\nDone. {features_df.shape[1]} columns total, {features_df.shape[0]} rows.")
    return features_df

## 10. Run feature extraction on the labeled splits

In [ ]:
# Train
train_df = pd.read_csv(TRAIN_CSV)
features_train = extract_features_for_df(train_df, name="train")
features_train.to_csv(os.path.join(OUTPUT_DIR, "features_train.csv"), index=False)
print(f"Saved features_train.csv ({len(features_train)} rows)")


Extracting features for: train
Input: 1104 sentences

Computing per-sentence features (basic, lexicon, bertje tokens)...


sentence-level:   0%|          | 0/1104 [00:00<?, ?it/s]


Running spaCy pipeline...


spaCy:   0%|          | 0/1104 [00:00<?, ?it/s]


Computing position features...

Computing context-relative features


...

Computing similarity to rest of blurb...


Batches:   0%|          | 0/18 [00:00<?, ?it/s]

Computing similarity to final_edit...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Computing word overlap with final_edit...

Computing blurb-relative features...

Done. 62 columns total, 1104 rows.
Saved features_train.csv (1104 rows)


In [ ]:
# Validation
val_df = pd.read_csv(VAL_CSV)
features_val = extract_features_for_df(val_df, name="val")
features_val.to_csv(os.path.join(OUTPUT_DIR, "features_val.csv"), index=False)
print(f"Saved features_val.csv ({len(features_val)} rows)")


Extracting features for: val
Input: 302 sentences

Computing per-sentence features (basic, lexicon, bertje tokens)...


sentence-level:   0%|          | 0/302 [00:00<?, ?it/s]


Running spaCy pipeline...


spaCy:   0%|          | 0/302 [00:00<?, ?it/s]


Computing position features...

Computing context-relative features


...

Computing similarity to rest of blurb...


Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Computing similarity to final_edit...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Computing word overlap with final_edit...

Computing blurb-relative features...

Done. 62 columns total, 302 rows.
Saved features_val.csv (302 rows)


In [ ]:
# Test
test_df = pd.read_csv(TEST_CSV)
features_test = extract_features_for_df(test_df, name="test")
features_test.to_csv(os.path.join(OUTPUT_DIR, "features_test.csv"), index=False)
print(f"Saved features_test.csv ({len(features_test)} rows)")


Extracting features for: test
Input: 481 sentences

Computing per-sentence features (basic, lexicon, bertje tokens)...


sentence-level:   0%|          | 0/481 [00:00<?, ?it/s]


Running spaCy pipeline...


spaCy:   0%|          | 0/481 [00:00<?, ?it/s]


Computing position features...

Computing context-relative features


...

Computing similarity to rest of blurb...


Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Computing similarity to final_edit...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Computing word overlap with final_edit...

Computing blurb-relative features...

Done. 62 columns total, 481 rows.
Saved features_test.csv (481 rows)


## 11. Run extraction on 25,000 of unlabeled data to later create weak labels on

In [ ]:
# Preparing the data and selecting a 25,000 sample
# Loading the data
unlabeled_df = pd.read_csv(UNLABELED_TSV, sep="\t")
print(f"Unlabeled shape: {unlabeled_df.shape}")
print(f"Columns: {unlabeled_df.columns.tolist()}")

# Dropping rows with missing sentence
n_before = len(unlabeled_df)
unlabeled_df = unlabeled_df.dropna(subset=[SENT_COL]).reset_index(drop=True)
print(f"After dropping NaN: {len(unlabeled_df)} sentences (removed {n_before - len(unlabeled_df)})")

# Getting a sample
unlabeled_sample = unlabeled_df.sample(n=25000, random_state=42)

Unlabeled shape: (235253, 6)
Columns: ['blurb_id', 'blurb', 'final_edit', 'blurb_sent', 'sent_position', 'unnecessary']
After dropping NaN: 235253 sentences (removed 0)


In [ ]:
# Creating and saving the features
features_unlabeled = extract_features_for_df(unlabeled_sample, name="unlabeled")
features_unlabeled.to_csv(os.path.join(OUTPUT_DIR, "features_unlabeled_sample.csv"), index=False)
print(f"Saved features_unlabeled_sample.csv ({len(features_unlabeled)} rows)")


Extracting features for: unlabeled
Input: 25000 sentences

Computing per-sentence features (basic, lexicon, bertje tokens)...


sentence-level:   0%|          | 0/25000 [00:00<?, ?it/s]


Running spaCy pipeline...


spaCy:   0%|          | 0/25000 [00:00<?, ?it/s]


Computing position features...

Computing context-relative features


...

Computing similarity to rest of blurb...


Batches:   0%|          | 0/391 [00:00<?, ?it/s]

Computing similarity to final_edit...


Batches:   0%|          | 0/239 [00:00<?, ?it/s]

Computing word overlap with final_edit...

Computing blurb-relative features...

Done. 62 columns total, 25000 rows.
Saved features_unlabeled_sample.csv (25000 rows)


## 12. Summary

In [ ]:
print("Feature columns:")
for col in features_train.columns:
    print(f"  {col}")
print(f"\nTotal features: {features_train.shape[1] - 3}  (excluding blurb_id, blurb_sent, label)")

print("\nFiles saved to:", OUTPUT_DIR)
for f in sorted(os.listdir(OUTPUT_DIR)):
    path = os.path.join(OUTPUT_DIR, f)
    size_mb = os.path.getsize(path) / 1024 / 1024
    print(f"  {f}  ({size_mb:.2f} MB)")

Feature columns:
  char_count
  word_count
  avg_word_len
  n_exclamation
  n_question
  n_quote_chars
  n_uppercase_words
  starts_with_quote
  ends_with_dash
  has_em_dash
  lex_promotional_hyperbolic_count
  lex_promotional_evaluative_count
  lex_intensifier_count
  lex_success_marker_count
  lex_award_keyword_count
  lex_comparison_marker_count
  lex_imperative_action_count
  lex_second_person_count
  lex_media_outlet_count
  lex_citation_marker_count
  lex_reader_experience_count
  lex_author_info_count
  lex_genre_marker_count
  n_bertje_tokens
  n_tokens_spacy
  n_adj
  n_adv
  n_verb
  n_aux
  n_noun
  n_propn
  n_pron
  adj_density
  verb_density
  aux_density
  noun_density
  max_adj_run
  n_adj_stacks
  starts_with_verb
  n_person
  n_org
  n_loc
  n_gpe
  n_norp
  n_event
  n_workofart
  n_misc
  total_ents
  sent_position
  rel_position
  is_first
  is_last
  sim_to_rest_of_blurb
  sim_to_final_edit
  word_overlap_final_edit
  adj_density_rel_to_blurb
  verb_density_rel_to